# Day 3 · Exercise 3: Raw REST — Call Ollama with requests

**What you'll build:** `ask_rest` — call your local model using only `requests.post()` and raw HTTP, with no LLM-specific library.

**Why it matters:** Every LLM library is just HTTP POST + JSON. Doing it once by hand means you truly understand what's happening underneath — and you can call any REST API regardless of whether a Python package exists for it.

**Prereq check:**
1. Ollama running with `llama3.2` pulled
2. `pip install requests` (usually already installed)

## Your Implementation

In [ ]:
import requests

MODEL   = "llama3.2"
API_URL = "http://localhost:11434/api/chat"

def ask_rest(question: str) -> str:
    """Call the local model using raw HTTP via the requests library.

    POST to http://localhost:11434/api/chat with a JSON body containing
    model, messages, and stream=False. Return the reply text.

    Args:
        question: The question to ask.

    Returns:
        The model's text reply as a string.

    Example:
        ask_rest("What is 2 + 2?")  ->  "4"
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _ollama_running():
    try:
        import urllib.request  # stdlib — no install needed
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(ask_rest), 'ask_rest is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: Ollama running + function returns a string
    if not _ollama_running():
        print(f'{_FAIL} Check 2/{total}: Ollama server is not running')
        print('  → macOS: open the Ollama app · Linux/Windows: ollama serve')
        return

    result = None
    try:
        result = ask_rest('Reply with only the word HELLO.')
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        print(f'{_PASS} Check 2/{total}: returns a string')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: call failed — {e}')
        print('  → URL should be http://localhost:11434/api/chat')
        print('  → Did you set  "stream": False  in the JSON body?')
        return

    if result is None:
        return

    # Check 3: result is non-empty
    try:
        assert len(result) > 0, 'returned an empty string'
        print(f'{_PASS} Check 3/{total}: result is non-empty')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: works with a different question
    try:
        r2 = ask_rest('What colour is the sky? One word.')
        assert isinstance(r2, str) and len(r2) > 0
        print(f'{_PASS} Check 4/{total}: works with a different question')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 3 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Extend `ask_rest` to also return the token counts alongside the text:
```python
def ask_rest_full(question: str) -> dict:
    # Return {"text": ..., "input_tokens": ..., "output_tokens": ...}
    # Hint: raw["prompt_eval_count"] and raw["eval_count"]
```
This maps directly to what `response.usage` gives you in the cloud APIs — same data, different key names.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import requests

MODEL   = "llama3.2"
API_URL = "http://localhost:11434/api/chat"

def ask_rest(question: str) -> str:
    response = requests.post(
        API_URL,
        json={
            "model":    MODEL,
            "messages": [{"role": "user", "content": question}],
            "stream":   False,
        }
    )
    response.raise_for_status()
    return response.json()["message"]["content"]
```

**Why this works:** `requests.post(url, json=...)` serialises the dict to JSON, sets `Content-Type: application/json`, and sends the HTTP POST. `raise_for_status()` raises an exception if the server returns an error code (4xx/5xx) rather than silently returning a broken response. `response.json()` parses the JSON body into a Python dict. The reply text is at `["message"]["content"]` — the same nested path as the native `ollama` package uses, because the raw response is what that package returns directly.
</details>